---
title: "DRG Cleaning (Python) v2"

author: "Carlos Resurreccion"

date: "2024-11-19"

---


In [ ]:
import os
import subprocess
import hashlib
from concurrent.futures import ThreadPoolExecutor
from tqdm import tqdm


def calculate_max_threads():
    """
    Dynamically calculate the optimal number of threads or processes based on CPU count.
    """
    max_threads = os.cpu_count()
    print(f"Detected {max_threads} logical cores. Using {max_threads} threads.")
    return max_threads


def calculate_file_hash(file_path):
    """
    Calculate the MD5 hash of a local file.
    """
    with open(file_path, "rb") as f:
        md5 = hashlib.md5()
        while chunk := f.read(8192):
            md5.update(chunk)
    return md5.hexdigest()


def get_gcs_hashes(bucket_path):
    """
    Get the MD5 hashes of all files in the GCS bucket using `gsutil hash`.
    """
    print("Fetching hashes for all GCS files...")
    result = subprocess.run(
        ["gsutil", "ls", "-r", bucket_path],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        print(f"Error listing files in GCS bucket: {result.stderr}")
        return {}

    gcs_files = {}
    file_list = [line.strip() for line in result.stdout.splitlines() if line.startswith(bucket_path)]

    for gcs_file in tqdm(file_list, desc="Calculating GCS Hashes", unit="file"):
        hash_result = subprocess.run(
            ["gsutil", "hash", gcs_file],
            capture_output=True,
            text=True,
        )
        if hash_result.returncode != 0:
            print(f"Error calculating hash for {gcs_file}: {hash_result.stderr}")
            continue

        # Extract MD5 hash from `gsutil hash` output
        for line in hash_result.stdout.splitlines():
            if "Hash (md5):" in line:
                gcs_md5 = line.split(":")[1].strip()
                gcs_files[gcs_file] = gcs_md5
                break

    return gcs_files


def upload_file_to_gcs(local_file_path, gcs_file_path):
    """
    Upload a local file to GCS.
    """
    print(f"Uploading {local_file_path} to {gcs_file_path}...")
    subprocess.run(["gsutil", "cp", local_file_path, gcs_file_path])


def download_file_from_gcs(gcs_file_path, local_file_path):
    """
    Download a file from GCS.
    """
    print(f"Downloading {gcs_file_path} to {local_file_path}...")
    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)
    subprocess.run(["gsutil", "cp", gcs_file_path, local_file_path])


def get_local_file_hashes(local_path, exclusions):
    """
    Get the file paths and hashes from the local directory using parallel processing.
    Excludes files listed in the exclusions.
    """
    files = {}
    file_paths = []
    for root, _, filenames in os.walk(local_path):
        for filename in filenames:
            full_path = os.path.join(root, filename)
            relative_path = full_path.replace(local_path + "/", "")
            if relative_path not in exclusions:
                file_paths.append(full_path)

    max_threads = calculate_max_threads()

    with ThreadPoolExecutor(max_threads) as executor, tqdm(total=len(file_paths), desc="Hashing Local Files", unit="file") as progress_bar:
        for file_path, file_hash in zip(file_paths, executor.map(calculate_file_hash, file_paths)):
            progress_bar.update(1)
            relative_path = file_path.replace(local_path + "/", "")
            files[relative_path] = file_hash
    return files


def sync_files(local_path, bucket_path, exclusions, upload=False, download=False):
    """
    Sync files between local and GCS bucket based on MD5 hash comparison.
    """
    if upload:
        print("Fetching local file hashes...")
        local_files = get_local_file_hashes(local_path, exclusions)
        print("Fetching GCS file hashes...")
        gcs_files = get_gcs_hashes(bucket_path)

        for relative_path, local_md5 in tqdm(local_files.items(), desc="Comparing for Upload", unit="file"):
            gcs_file_path = f"{bucket_path}/{relative_path}"
            gcs_md5 = gcs_files.get(gcs_file_path)

            if gcs_md5 is None:
                print(f"{relative_path} missing in GCS. Uploading...")
                upload_file_to_gcs(os.path.join(local_path, relative_path), gcs_file_path)
            elif gcs_md5 != local_md5:
                print(f"MD5 mismatch for {relative_path}. Re-uploading...")
                upload_file_to_gcs(os.path.join(local_path, relative_path), gcs_file_path)

    if download:
        print("Fetching GCS file hashes...")
        gcs_files = get_gcs_hashes(bucket_path)
        print("Fetching local file hashes...")
        local_files = get_local_file_hashes(local_path, exclusions)

        for gcs_file_path, gcs_md5 in tqdm(gcs_files.items(), desc="Comparing for Download", unit="file"):
            relative_path = os.path.relpath(gcs_file_path, bucket_path)
            if relative_path not in exclusions:
                local_md5 = local_files.get(relative_path)

                if local_md5 is None:
                    print(f"{relative_path} missing locally. Downloading...")
                    download_file_from_gcs(gcs_file_path, os.path.join(local_path, relative_path))
                elif gcs_md5 != local_md5:
                    print(f"MD5 mismatch for {relative_path}. Re-downloading...")
                    download_file_from_gcs(gcs_file_path, os.path.join(local_path, relative_path))


# Main Execution
bucket_path = "gs://phic-claims-checkpoints/temp/data"
vm_path = "/mnt/data-disk/data"
desktop_path = "/home/data"

# Files to Exclude
exclusions = [
    "claims/raw/claims_extract_CLAIMS 2018.csv",
    "claims/raw/claims_extract_CLAIMS 2019.csv",
    "claims/raw/claims_extract_CLAIMS 2020.csv",
    "claims/raw/claims_extract_CLAIMS 2021.csv",
    "claims/raw/claims_extract_CLAIMS 2022.tsv",
    "claims/raw/claims_extract_CLAIMS 2023.tsv",
]

# Flags
to_upload_from_vm = False  # Set True to upload from VM to GCS
to_upload_from_desktop = False  # Set True to upload from Desktop to GCS
to_download_to_vm = False  # Set True to download from GCS to VM
to_download_to_desktop = True  # Set True to download from GCS to Desktop

if to_upload_from_vm:
    print("Syncing VM to GCS...")
    sync_files(vm_path, bucket_path, exclusions, upload=True)

if to_upload_from_desktop:
    print("Syncing Desktop to GCS...")
    sync_files(desktop_path, bucket_path, exclusions, upload=True)

if to_download_to_vm:
    print("Syncing GCS to VM...")
    sync_files(vm_path, bucket_path, exclusions, download=True)

if to_download_to_desktop:
    print("Syncing GCS to Desktop...")
    sync_files(desktop_path, bucket_path, exclusions, download=True)

print("Completed.")

In [ ]:
# Initialize variables
thread_offset = 0
sample_size_divisor = 625

# Whether to sample each split_part by sample_size_divisor
# (useful when iterating through code runs in quick succession)
to_sample = False  # Flag to indicate sampling
to_write = True    # Flag to enable writing outputs
to_flush = False   # Flag to enable flushing buffers
to_parallel = True # Flag for enabling parallel processing
to_debug = False   # Flag for enabling debugging

# Display the parallelization status
print("Parallelization:", to_parallel, "\n")

# Set verbose output based on debugging flag
verbose_output = True if to_debug else False

# Path to the year_to_load file
year_file_path = "/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/cache/year_to_load.txt"

# Read the year from the file
with open(year_file_path, "r") as file:
    year_to_load = file.read().strip()  # .strip() removes any surrounding whitespace or newlines

# Create a suffix based on the `to_sample` flag and sample_size_divisor
if to_sample:
    suffix = f"_sampled_{sample_size_divisor}_"
else:
    suffix = "_full_"

In [ ]:
import pandas as pd
import os
import numpy as np
from grouper import seeker
from multiprocessing import Pool, cpu_count
import traceback
import swifter
import traceback
import sys
import io
import pyarrow
import gc

In [ ]:
# Construct the file path for the Feather file
feather_file_path = f"~/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_7_py_input/python_input_{year_to_load}{suffix}.feather"

# Expand the `~` to the user's home directory
feather_file_path = os.path.expanduser(feather_file_path)

# Read the Feather file
pandas_df = pd.read_feather(feather_file_path)

# Print the DataFrame or process it as needed
# print(pandas_df)
print(pandas_df[(pandas_df['patage'] == 0) & (pandas_df['ageday'].notna())])

In [ ]:
# # Define the patient data as a dictionary with lists
# pat = {
#     'id_series': ['1'],
#     'patage': ["0"],
#     'patsex': ['M'],
#     'date_adm': ['2018-03-02 23:21:00'],
#     'date_dis': ['2018-03-10 04:05:00'],
#     'pdx': ['P239'],
#     'sdx1': ['Z380'],
#     'sdx2': [None],
#     'sdx3': [None],
#     'sdx4': [None],
#     'sdx5': [None],
#     'sdx6': [None],
#     'sdx7': [None],
#     'sdx8': [None],
#     'sdx9': [None],
#     'sdx10': [None],
#     'sdx11': [None],
#     'sdx12': [None],
#     'proc1': ['9929'],
#     'proc2': ['9933'],
#     'proc3': ['9959'],
#     'proc4': [None],
#     'proc5': [None],
#     'proc6': [None],
#     'proc7': [None],
#     'proc8': [None],
#     'proc9': [None],
#     'proc10': [None],
#     'proc11': [None],
#     'proc12': [None],
#     'proc13': [None],
#     'proc14': [None],
#     'proc15': [None],
#     'proc16': [None],
#     'proc17': [None],
#     'proc18': [None],
#     'proc19': [None],
#     'proc20': [None],
#     'discharge': [1],
#     'birthweight': [2.732],
#     'ageday': [0]
# }
# pandas_df = pd.DataFrame(pat)

In [ ]:
# Convert the column types explicitly
print("Converting data types")
pandas_df['patage'] = pd.to_numeric(pandas_df['patage'], errors='coerce')
pandas_df['ageday'] = pd.to_numeric(pandas_df['ageday'], errors='coerce')
pandas_df['birthweight'] = pd.to_numeric(pandas_df['birthweight'], errors='coerce')
pandas_df['discharge'] = pandas_df['discharge'].astype('Int64')

# Convert string columns to 'string' dtype and replace NA values with None
string_columns = ['id_series', 'patsex', 'pdx', 'sdx1', 'sdx2', 'sdx3', 'sdx4', 'sdx5', 'sdx6', 'sdx7', 'sdx8', 'sdx9', 'sdx10', 'sdx11', 'sdx12',
                  'proc1', 'proc2', 'proc3', 'proc4', 'proc5', 'proc6', 'proc7', 'proc8', 'proc9', 'proc10', 'proc11', 'proc12',
                  'proc13', 'proc14', 'proc15', 'proc16', 'proc17', 'proc18', 'proc19', 'proc20', 'date_adm', 'date_dis']

print("Replacing with None")
# Replace missing values in place
pandas_df.replace([pd.NA, np.nan, '<NA>', 'None', 'NA', -2147483648], None, inplace=True)

print("Converting to string")
# Convert columns to string dtype after replacing the values
pandas_df[string_columns] = pandas_df[string_columns].astype('string')

print("Replacing -2147483648 with None")
# Replace -2147483648 with None again (in case it was missed)
pandas_df.replace(-2147483648, None, inplace=True)

# print("Filter discharge")
# # Filter rows where 'discharge' is not in [1, 2, 3, 4, 9]
# not_in_list_values = pandas_df.loc[~pandas_df['discharge'].isin([1, 2, 3, 4, 9]), 'discharge']

# # Get unique values and their counts
# unique_not_in_list_values = not_in_list_values.value_counts()

# # Print the unique values and their counts
# print(unique_not_in_list_values)

print("Generating info()")
pandas_df.info()
print(pandas_df)

In [ ]:
print(pandas_df[(pandas_df['patage'] == 0) & (pandas_df['ageday'].notna())])

In [ ]:
# Sample 100,000 rows from the DataFrame
# full_df = pandas_df
# pandas_df = pandas_df.sample(n=100000, random_state=42)

In [ ]:
# # SINGLE THREADED-VERSION
# print("Initializing Libraries")
# # Initialize the necessary libraries
# libs = seeker.Libraries()

# # Define a function to instantiate a Patient object for each row
# def process_patient(row, libs):
#     try:
#         # Convert the row to a dictionary and create a Patient object
#         patient = seeker.Patient(row.to_dict(), libs)
        
#         # Extract relevant attributes from the Patient object
#         result = {
#             'mdc': patient.mdc,
#             'pdc': patient.pdc,
#             'pccl': patient.pccl,
#             'drg': patient.drg,
#             'error_code': patient.error_code,
#             'warning_code': patient.warning_code
#         }
        
#         return pd.Series(result)
    
#     except Exception as e:
#         # Log the error and row information for debugging
#         print(f'''Error processing patient with id_series {row['id_series']}: {e}''')
        
#         # Optionally, you can log more information such as row content or traceback
#         traceback.print_exc()  # Print the full stack trace for more details
        
#         # Return None or default values for the error case
#         return pd.Series({
#             'mdc': None,
#             'pdc': None,
#             'pccl': None,
#             'drg': None,
#             'error_code': None,
#             'warning_code': None
#         })

# print("swifter.apply process_patient")
# # Apply the Patient class directly to each row using swifter
# pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = pandas_df.apply(
#     lambda row: process_patient(row, libs),
#     axis=1
# )
# print("Renaming columns")
# # Store the result in output to be retrieved by R
# output = pandas_df.rename(columns={'drg': 'py_drg'})
# print("Reordering columns")
# # Define the desired column order
# desired_columns = [
#     'id_series', 'mdc', 'pdc', 
#     'pccl', 'py_drg', 'error_code', 
#     'warning_code'
# ]
# print("Subsetting columns")
# # Reorder the DataFrame and drop any columns not in the desired list
# output = output[desired_columns]
# print(output)

In [ ]:
# Define a function to instantiate a Patient object for each row
# def process_patient(row, libs):
#     try:
#         # Convert the row to a dictionary and create a Patient object
#         patient = seeker.Patient(row.to_dict(), libs)

#         # Extract relevant attributes from the Patient object
#         result = {
#             'mdc': patient.mdc,
#             'pdc': patient.pdc,
#             'pccl': patient.pccl,
#             'drg': patient.drg,
#             'error_code': patient.error_code,
#             'warning_code': patient.warning_code
#         }

#         return result

#     except Exception as e:
#         # Log the error and row information for debugging
#         print(f"Error processing patient with id_series {row['id_series']}: {e}")
#         traceback.print_exc()

#         # Return default values for error cases
#         return {
#             'mdc': None,
#             'pdc': None,
#             'pccl': None,
#             'drg': None,
#             'error_code': None,
#             'warning_code': None
#         }

# # Initialize the necessary libraries
# print("Initializing Libraries")
# libs = seeker.Libraries()

# # Split DataFrame into chunks for multiprocessing
# num_cores = cpu_count()  # Automatically detect the number of CPU cores
# chunks = np.array_split(pandas_df, num_cores)  # Split the DataFrame into chunks

# print(f"Processing using {num_cores} cores")

# # Use multiprocessing to process each chunk in parallel
# with Pool(num_cores) as pool:
#     results = pool.starmap(process_chunk, [(chunk, libs) for chunk in chunks])

# # Combine the results back into a single DataFrame
# processed_df = pd.concat(results)

# # Add the processed columns to the original DataFrame
# pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = processed_df

# # Rename and reorder columns
# pandas_df.rename(columns={'drg': 'py_drg'}, inplace=True)
# desired_columns = [
#     'id_series', 'mdc', 'pdc',
#     'pccl', 'py_drg', 'error_code',
#     'warning_code'
# ]
# # Drop columns not in the desired list
# columns_to_drop = [col for col in pandas_df.columns if col not in desired_columns]
# pandas_df.drop(columns=columns_to_drop, inplace=True)
# Initialize the necessary libraries

In [ ]:
def process_patient(row, libs):
    # Convert the row to a dictionary and create a Patient object
    patient = seeker.Patient(row.to_dict(), libs)

    # Extract relevant attributes from the Patient object
    result = {
        'mdc': patient.mdc,
        'pdc': patient.pdc,
        'pccl': patient.pccl,
        'drg': patient.drg,
        'error_code': patient.error_code,
        'warning_code': patient.warning_code
    }

    return result
# Function to process a chunk of the DataFrame
def process_chunk(chunk, libs):
    return chunk.apply(lambda row: pd.Series(process_patient(row, libs)), axis=1)

print("Initializing Libraries")
libs = seeker.Libraries()

# Split DataFrame into chunks
chunks = np.array_split(pandas_df, cpu_count())
del pandas_df  # Free up memory for the original DataFrame
gc.collect()

print(f"Processing using {len(chunks)} cores")

# Use multiprocessing to process each chunk in parallel
with Pool(len(chunks)) as pool:
    # Process all chunks in parallel and collect results at once
    results = pool.starmap(process_chunk, [(chunk, libs) for chunk in chunks])

# Concatenate results into a single DataFrame
processed_df = pd.concat(results, ignore_index=True)
del results  # Free memory for intermediate results
gc.collect()

# Add the processed columns to the original DataFrame
pandas_df[['mdc', 'pdc', 'pccl', 'drg', 'error_code', 'warning_code']] = processed_df
del processed_df  # Free up memory for the processed DataFrame
gc.collect()

# Rename and reorder columns
pandas_df.rename(columns={'drg': 'py_drg'}, inplace=True)

# Drop unwanted columns
desired_columns = [
    'id_series', 'mdc', 'pdc',
    'pccl', 'py_drg', 'error_code',
    'warning_code'
]
columns_to_drop = [col for col in pandas_df.columns if col not in desired_columns]
pandas_df.drop(columns=columns_to_drop, inplace=True)
gc.collect()

In [ ]:
# Construct the file path
file_path = f"/home/resurreccion_cmc_gmail_com/drg-pipeline/data-cleaning/data/checkpoints/checkpoint_8_py_output/python_output_{year_to_load}{suffix}.feather"
# Save the DataFrame as a Feather file
pandas_df.to_feather(file_path)